In [1]:
import cv2 as cv
import dlib
import os
import glob
import numpy as np
from tqdm import tqdm

In [2]:
# 实例化面部检测对象
detector =dlib.get_frontal_face_detector()

In [3]:
# 实例化特征识别对象
predictor = dlib.shape_predictor("./model/shape_predictor_68_face_landmarks.dat")

In [4]:
def collectData(root):
    # 存储每个面部图像的 68 个特征点以及对应的面部面积
    data = np.empty([0,68,2])
    area = np.empty([0])
    
    files = glob.glob(os.path.join(root,'*.jpg'))

    for i in tqdm(range(len(files))):
        imgFile = files[i]
        frame = cv.imread(imgFile)
        img = cv.cvtColor(frame,cv.COLOR_BGR2RGB)
        dets = detector(img,1)

        d = dets[0] # 面部矩形框
        shape = predictor(img,d)

        # 绝对坐标
        oriX = d.left()
        oriY = d.top()
        facePoints = np.empty([0,2]) # 初始shape

        # 遍历 68 个面部特征点并计算相对坐标
        for i in range(68):
            point = np.array([shape.part(i).x - oriX,shape.part(i).y - oriY])
            # 扩增维度
            point = np.expand_dims(point,axis=0)
            # 合并面部点坐标数组 此时 facePoints 的 shape 为 (68,2)
            facePoints = np.concatenate((facePoints,point),axis=0)

        # 表示所有面部特征的数组 data 的 shape 为 (0,68,2) 因此扩增 1 个维度
        facePoints = np.expand_dims(facePoints,axis=0)
        # 面部特征点合并
        data = np.concatenate((data,facePoints),axis=0)
        # 将当前面部的面积合并到总面部数组
        area = np.append(area, d.area())
    return data,area

In [5]:
openFace,openFaceArea = collectData('./data/OpenFace')

100%|██████████| 1226/1226 [02:31<00:00,  8.09it/s]


In [6]:
closedFace,closedFaceArea = collectData('./data/ClosedFace')

100%|██████████| 1002/1002 [01:52<00:00,  8.91it/s]


In [7]:
print(openFace.shape)
print(openFaceArea.shape)
print(closedFace.shape)
print(closedFaceArea.shape)

(1226, 68, 2)
(1226,)
(1002, 68, 2)
(1002,)


In [8]:
np.save('./data/openFace.npy',openFace)
np.save('./data/openFaceArea.npy',openFaceArea)
np.save('./data/closedFace.npy',closedFace)
np.save('./data/closedFaceArea.npy',closedFaceArea)